In [0]:
# ----------------------------------------
# Step 1: Append New Data (Fixed)
# ----------------------------------------

gold_path = "workspace.ecommerce.gold_user_purchase_predictions"

from pyspark.sql.functions import current_date

new_df = spark.table(gold_path) \
    .limit(5) \
    .withColumn("scoring_date", current_date())   # ✅ correct column name

new_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(gold_path)

# Check Delta history
spark.sql(f"""
DESCRIBE HISTORY {gold_path}
""").show(truncate=False)

In [0]:
# ----------------------------------------
# Step 2: Read Old Version
# ----------------------------------------

old_df = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .table(gold_path)

old_df.show()

In [0]:
# ----------------------------------------
# Step 3: Compare Versions
# ----------------------------------------

latest_df = spark.read.format("delta").table(gold_path)

latest_df.subtract(old_df).show()